In [94]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import joblib

# 1. 허깅페이스에서 데이터셋 다운로드 및 로드
dataset = load_dataset("c01dsnap/CIC-IDS2017")

# 2. 분석 및 전처리를 위해 Pandas DataFrame으로 변환
df = dataset['train'].to_pandas()

# 데이터 확인
print(df.shape)  # (2830743, 79) - 약 283만 개 행 확인!
print(df[' Label'].value_counts())  # 라벨별 데이터 개수 확인

(2830743, 79)
 Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [79]:
df.isnull().sum()

 Destination Port              0
 Flow Duration                 0
 Total Fwd Packets             0
 Total Backward Packets        0
Total Length of Fwd Packets    0
                              ..
Idle Mean                      0
 Idle Std                      0
 Idle Max                      0
 Idle Min                      0
 Label                         0
Length: 79, dtype: int64

In [95]:
# 공백 제거
df.columns = df.columns.str.strip()

# 무한대 값(inf, -inf)을 결측치(NaN)로 변환
df.replace([np.inf,-np.inf],np.nan,inplace=True)

# 결측치가 있는 행 제거
df.dropna( inplace=True)

# 'BENIGN'이면 0, 그 외 모든 공격은 1로 매핑
# 정상(0) : 2,271,320개
# 악성(1) : 556556개
df['Binary_Label'] = np.where(df['Label'] == 'BENIGN', 0, 1)

# 제거할 열 
# 데이터의 값이 전부 0이거나 학습에 큰 영향을 미치지 않는 열
drop_cols = [
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate",
    "Bwd PSH Flags",
    "Bwd URG Flags"
]

df.drop(columns=drop_cols, inplace=True)

In [96]:
# 2827876개 72열
print(f"최종 데이터 크기 : {df.shape}")

최종 데이터 크기 : (2827876, 72)


In [97]:
# Feature 데이터 분리
X = df.drop(columns=["Label", "Binary_Label"])

# Label 데이터 분리
y = df["Binary_Label"]

np.save("X.npy", X.to_numpy())
np.save("y.npy", y.to_numpy())

In [98]:
joblib.dump(X.columns.tolist(), "feature_columns.pkl")

['feature_columns.pkl']

In [62]:
# 1. 무한대 값(inf, -inf)을 결측치(NaN)로 변환
df.replace([np.inf,-np.inf],np.nan,inplace=True)

# 2. 결측치가 있는 행 제거
df.dropna(inplace=True)

In [63]:
# 'BENIGN'이면 0, 그 외 모든 공격은 1로 매핑
# 정상(0) : 2,271,320개
# 악성(1) : 556556개
df['Binary_Label'] = np.where(df['Label'] == 'BENIGN', 0, 1)

In [64]:
# 정상과 공격 비율 최종 확인
print(df['Binary_Label'].value_counts())

Binary_Label
0    2271320
1     556556
Name: count, dtype: int64


In [85]:
df.info()

<class 'pandas.DataFrame'>
Index: 2827876 entries, 0 to 2830742
Data columns (total 73 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int64  
 1   Flow Duration                int64  
 2   Total Fwd Packets            int64  
 3   Total Backward Packets       int64  
 4   Total Length of Fwd Packets  int64  
 5   Total Length of Bwd Packets  int64  
 6   Fwd Packet Length Max        int64  
 7   Fwd Packet Length Min        int64  
 8   Fwd Packet Length Mean       float64
 9   Fwd Packet Length Std        float64
 10  Bwd Packet Length Max        int64  
 11  Bwd Packet Length Min        int64  
 12  Bwd Packet Length Mean       float64
 13  Bwd Packet Length Std        float64
 14  Flow Bytes/s                 float64
 15  Flow Packets/s               float64
 16  Flow IAT Mean                float64
 17  Flow IAT Std                 float64
 18  Flow IAT Max                 int64  
 19  Flow IAT Min    

In [45]:
df

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2830738,53,32215,4,2,112,152,28,28,28.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830739,53,324,2,2,84,362,42,42,42.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830740,58030,82,2,1,31,6,31,0,15.5,21.92031,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830741,53,1048635,6,2,192,256,32,32,32.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [48]:
df.duplicated().sum()

KeyboardInterrupt: 

In [50]:
duplicate_rate = df.duplicated().mean() * 100
print(duplicate_rate)

10.893994968812075


In [52]:
duplicates = df[df.duplicated(keep=False)]

print(duplicates.head(20))

      Destination Port   Flow Duration   Total Fwd Packets  \
76                  21              50                   1   
109                 22             187                   1   
111                 22             111                   1   
384                443             161                   2   
386                465              49                   2   
387                 80           50486                   1   
390                 80           50641                   1   
405                137              22                  13   
422                 53             259                   2   
423                 53             199                   2   
426                 53             198                   2   
430                 53             262                   2   
434                 53             271                   2   
441                 53             301                   2   
443                443               4                   2   
454     

In [53]:
print(df.duplicated().value_counts())

False    2522362
True      308381
Name: count, dtype: int64


In [66]:
# 완전히 중복된 행 표시
duplicate_mask = df.duplicated()

# 중복 행만 추출
duplicate_rows = df.loc[duplicate_mask]

# 원본 Label별 중복 개수
print(duplicate_rows["Label"].value_counts())

Label
BENIGN                      176263
PortScan                     68110
DoS Hulk                     57278
SSH-Patator                   2678
FTP-Patator                   2004
DoS slowloris                  411
DoS Slowhttptest               271
Web Attack � Brute Force        37
DDoS                            11
Bot                              8
DoS GoldenEye                    7
Name: count, dtype: int64


In [82]:
# 삭제해도 되는 열인지 확인
bulk_cols = [
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate"
]

print(df[bulk_cols].describe())

for col in bulk_cols:
    print(col, df[col].nunique())

# 결과를 보니 약280만개의 데이터가 0으로 제거해도 문제 없는 열

KeyError: "None of [Index(['Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate',\n       'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate'],\n      dtype='str')] are in the [columns]"

In [84]:
check_cols = [
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "CWE Flag Count",
    "ECE Flag Count"
]

print(df[check_cols].describe())

for col in check_cols:
    print(col, df[col].nunique())

       Bwd PSH Flags  Bwd URG Flags  CWE Flag Count  ECE Flag Count
count      2827876.0      2827876.0    2.827876e+06    2.827876e+06
mean             0.0            0.0    1.113910e-04    2.436458e-04
std              0.0            0.0    1.055361e-02    1.560726e-02
min              0.0            0.0    0.000000e+00    0.000000e+00
25%              0.0            0.0    0.000000e+00    0.000000e+00
50%              0.0            0.0    0.000000e+00    0.000000e+00
75%              0.0            0.0    0.000000e+00    0.000000e+00
max              0.0            0.0    1.000000e+00    1.000000e+00
Bwd PSH Flags 1
Bwd URG Flags 1
CWE Flag Count 2
ECE Flag Count 2


In [76]:
df.drop(columns=bulk_cols, inplace=True)

In [ ]:
# 추가로 제거해도 되는 열인지 확인
df["Fwd Header Length"].equals(df["Fwd Header Length.1"])

KeyError: 'Fwd Header Length.1'

In [77]:
df.drop(columns=["Fwd Header Length.1"], inplace=True)

KeyError: "['Fwd Header Length.1'] not found in axis"